# HH-RLHF-Style Evaluation: DPO vs SFT

Generate responses for the full 300-prompt HH-style subset, build DPO-vs-SFT pairs, optionally judge them with Kaggle Models, and preview a few examples side by side.


## Setup


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
REPO_DIR = Path('/kaggle/working/qwen-dpo-alignment')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}


In [ ]:
# Kaggle already has torch installed. Avoid reinstalling torch.
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps


## Paths


In [ ]:
# If checkpoints are attached from a training notebook, set these to /kaggle/input/... paths.
SFT_MODEL_PATH = "outputs/sft-qwen-0.5b/best"
DPO_MODEL_PATH = "outputs/dpo-qwen-0.5b/best"

# Examples for attached Kaggle training outputs:
# SFT_MODEL_PATH = "/kaggle/input/YOUR_TRAINING_OUTPUT_DATASET/outputs/sft-qwen-0.5b/best"
# DPO_MODEL_PATH = "/kaggle/input/YOUR_TRAINING_OUTPUT_DATASET/outputs/dpo-qwen-0.5b/best"

HH_EVAL_PATH = "eval_sets/hh_rlhf_eval_300_seed42.json"
N_EXAMPLES = None  # None means use the full HH eval subset
OUT_DIR = Path("outputs/hh_full_compare")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("SFT_MODEL_PATH:", SFT_MODEL_PATH)
print("DPO_MODEL_PATH:", DPO_MODEL_PATH)
print("HH_EVAL_PATH:", HH_EVAL_PATH)


## Generate Responses From Each Model


In [ ]:
import subprocess, sys, torch

def launch_generation(model_path, output_name, generator_name, device):
    cmd = [
        sys.executable, "scripts/generate_alpaca.py",
        "--model_name_or_path", str(model_path),
        "--subset_path", str(HH_EVAL_PATH),
        "--output_path", str(OUT_DIR / output_name),
        "--generator_name", generator_name,
        "--batch_size", "2",
        "--max_new_tokens", "256",
        "--device", device,
    ]
    if N_EXAMPLES is not None:
        cmd.extend(["--limit", str(N_EXAMPLES)])
    print("Launching:", " ".join(cmd))
    return subprocess.Popen(cmd)

sft_output = "sft_hh_eval_300.json" if N_EXAMPLES is None else f"sft_hh_eval_{N_EXAMPLES}.json"
dpo_output = "dpo_hh_eval_300.json" if N_EXAMPLES is None else f"dpo_hh_eval_{N_EXAMPLES}.json"

if torch.cuda.device_count() >= 2:
    jobs = [
        launch_generation(SFT_MODEL_PATH, sft_output, "qwen2.5-0.5b-sft", "cuda:0"),
        launch_generation(DPO_MODEL_PATH, dpo_output, "qwen2.5-0.5b-dpo", "cuda:1"),
    ]
    exit_codes = [job.wait() for job in jobs]
    if any(code != 0 for code in exit_codes):
        raise RuntimeError(f"Generation failed with exit codes: {exit_codes}")
else:
    print("Only one GPU/CPU visible; generating sequentially.")
    for model_path, output_name, generator_name in [
        (SFT_MODEL_PATH, sft_output, "qwen2.5-0.5b-sft"),
        (DPO_MODEL_PATH, dpo_output, "qwen2.5-0.5b-dpo"),
    ]:
        job = launch_generation(model_path, output_name, generator_name, "cuda:0" if torch.cuda.is_available() else "cpu")
        code = job.wait()
        if code != 0:
            raise RuntimeError(f"Generation failed for {generator_name} with exit code {code}")


## Build Pairwise File


In [ ]:
PAIR_PATH = OUT_DIR / ("dpo_vs_sft_hh_pairs_300.jsonl" if N_EXAMPLES is None else f"dpo_vs_sft_hh_pairs_{N_EXAMPLES}.jsonl")
!python scripts/build_pairwise_eval.py \
  --a_outputs {OUT_DIR}/{dpo_output} \
  --b_outputs {OUT_DIR}/{sft_output} \
  --a_name dpo \
  --b_name sft \
  --output_path {PAIR_PATH}


## Side-by-Side Preview


In [ ]:
import json
from IPython.display import Markdown, display

PREVIEW_N = 10
sft_rows = {row['id']: row for row in json.load(open(OUT_DIR / sft_output))}
dpo_rows = {row['id']: row for row in json.load(open(OUT_DIR / dpo_output))}
ids = list(dpo_rows.keys())[:PREVIEW_N]

def esc(text):
    return str(text or '').replace('`', '\`')

for i, item_id in enumerate(ids, start=1):
    dpo = dpo_rows[item_id]
    sft = sft_rows[item_id]
    md = f"""
## Preview {i}. {item_id}

**Category:** `{esc(dpo.get('category'))}`  \n
**Expected behavior:** {esc(dpo.get('expected_behavior'))}

**Instruction**

{esc(dpo['instruction'])}

### SFT

{esc(sft['output'])}

### DPO

{esc(dpo['output'])}

**Manual winner:** `________`

---
"""
    display(Markdown(md))


## Judge Full DPO-vs-SFT HH Set With Kaggle Models


In [ ]:
JUDGED_PATH = OUT_DIR / ("dpo_vs_sft_hh_judged_300.jsonl" if N_EXAMPLES is None else f"dpo_vs_sft_hh_judged_{N_EXAMPLES}.jsonl")
!python scripts/judge_pairwise_kaggle.py \
  --pairs_path {PAIR_PATH} \
  --output_path {JUDGED_PATH} \
  --judge_model google/gemini-3.6-flash \
  --batch_size 5 \
  --sleep_seconds 5 \
  --max_retries 3 \
  --reuse_judgments
